# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Srilaya30/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [25]:
import duckdb
from google.colab import userdata

# Read Hugging Face token from Colab Secrets
HF_TOKEN = userdata.get("HF_TOKEN")

# Connect to DuckDB
con = duckdb.connect()

# Create Hugging Face secret
con.execute(f"""
CREATE OR REPLACE SECRET hf_secret (
    TYPE HUGGINGFACE,
    TOKEN '{HF_TOKEN}'
);
""")

print("✅ Connected to DuckDB")

✅ Connected to DuckDB


In [26]:
REL = "hf://datasets/FlyRank/internship-warehouse"

con.sql(f"""
SELECT COUNT(*)
FROM read_parquet(
'{REL}/fact_content_daily_performance/month=2026-03/*.parquet'
)
""")

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│      9841378 │
└──────────────┘

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [27]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
REL = "hf://datasets/FlyRank/internship-warehouse"

con.sql(f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT content_hash_id) AS unique_content,
    MIN(report_date) AS first_date,
    MAX(report_date) AS last_date
FROM read_parquet(
'{REL}/fact_content_daily_performance/month=2026-03/*.parquet'
)
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────┬────────────────┬────────────┬────────────┐
│ total_rows │ unique_content │ first_date │ last_date  │
│   int64    │     int64      │    date    │    date    │
├────────────┼────────────────┼────────────┼────────────┤
│    9841378 │         331437 │ 2026-03-01 │ 2026-03-31 │
└────────────┴────────────────┴────────────┴────────────┘

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

## Features
- gsc_clicks
- gsc_impressions
- gsc_ctr
- gsc_avg_position
- ga4_pageviews

## Label / Proxy
Rank webpages for refresh priority based on historical search performance.

## Context
- client_hash_id
- content_hash_id
- report_date

These fields are used for identification, grouping, and filtering, not as model features.

## Excluded
- Future performance metrics
- Any label-derived fields
- Trend or outcome columns calculated using future data

These are excluded because they would cause data leakage.

In [28]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [29]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [30]:
con.sql(f"""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    COUNT(*) AS cnt
FROM read_parquet(
'{REL}/fact_content_daily_performance/month=2026-03/*.parquet'
)
GROUP BY report_date, client_hash_id, content_hash_id
HAVING COUNT(*) > 1
LIMIT 5
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────────┬────────────────┬─────────────────┬───────┐
│ report_date │ client_hash_id │ content_hash_id │  cnt  │
│    date     │    varchar     │     varchar     │ int64 │
├─────────────┴────────────────┴─────────────────┴───────┤
│                         0 rows                         │
└────────────────────────────────────────────────────────┘

In [31]:
con.sql(f"""
SELECT
    COUNT(*) AS row_count,
    MIN(report_date) AS first_date,
    MAX(report_date) AS last_date
FROM read_parquet(
'{REL}/fact_content_daily_performance/month=2026-03/*.parquet'
)
""")

┌───────────┬────────────┬────────────┐
│ row_count │ first_date │ last_date  │
│   int64   │    date    │    date    │
├───────────┼────────────┼────────────┤
│   9841378 │ 2026-03-01 │ 2026-03-31 │
└───────────┴────────────┴────────────┘

In [32]:
con.sql(f"""
SELECT
    COUNT(*) AS available_rows
FROM read_parquet(
'{REL}/fact_content_daily_performance/month=2026-03/*.parquet'
)
WHERE ga4_data_available IS TRUE
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────────┐
│ available_rows │
│     int64      │
├────────────────┤
│         413966 │
└────────────────┘

In [33]:
con.sql(f"""
SELECT *
FROM read_parquet(
'{REL}/fact_content_daily_performance/month=2026-03/*.parquet'
)
LIMIT 1
""")

┌─────────────┬─────────────────────────┬──────────────────────────┬────────────────┬────────────────┬────────────────────┬────────────────────┬─────────────────┬────────────┬──────────────────┬──────────────────┬───────────────┬──────────────┬───────────┬──────────────────────┬──────────────────────────┬──────────────────┬─────────────────┬───────────────────┬─────────────────┬───────────────┬─────────────┬────────────┬───────────────┬───────────┬────────────┬───────────┬─────────┬──────────┬───────────────┬─────────┐
│ report_date │     client_hash_id      │     content_hash_id      │ client_has_gsc │ client_has_ga4 │ gsc_data_available │ ga4_data_available │ gsc_impressions │ gsc_clicks │ gsc_sum_position │ gsc_avg_position │ ga4_pageviews │ ga4_sessions │ ga4_users │ ga4_engaged_sessions │ ga4_total_engagement_sec │ sessions_organic │ sessions_direct │ sessions_referral │ sessions_social │ sessions_paid │ sessions_ai │ ai_chatgpt │ ai_perplexity │ ai_gemini │ ai_copilot │ ai_claude

In [34]:
feature_frame = con.sql(f"""
SELECT
    gsc_impressions,
    gsc_clicks,
    CASE
        WHEN gsc_impressions > 0
        THEN gsc_clicks * 1.0 / gsc_impressions
        ELSE 0.0
    END AS gsc_ctr,
    gsc_avg_position,
    ga4_pageviews,
    ga4_sessions
FROM read_parquet(
'{REL}/fact_content_daily_performance/month=2026-03/*.parquet'
)
WHERE gsc_data_available IS TRUE
  AND ga4_data_available IS TRUE
LIMIT 10
""")

feature_frame

┌─────────────────┬────────────┬───────────────────────┬────────────────────┬───────────────┬──────────────┐
│ gsc_impressions │ gsc_clicks │        gsc_ctr        │  gsc_avg_position  │ ga4_pageviews │ ga4_sessions │
│      int64      │   int64    │        double         │       double       │     int64     │    int64     │
├─────────────────┼────────────┼───────────────────────┼────────────────────┼───────────────┼──────────────┤
│               5 │          0 │                   0.0 │                5.4 │             1 │            1 │
│              39 │          0 │                   0.0 │  5.666666666666667 │             2 │            2 │
│             179 │          0 │                   0.0 │  5.156424581005586 │             2 │            2 │
│              72 │          0 │                   0.0 │  7.694444444444445 │             1 │            1 │
│            3282 │          1 │ 0.0003046922608165753 │  6.167885435709933 │             1 │            1 │
│              39 │

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

## Data Limits

This dataset supports decision-making using historical search and analytics data, but it cannot measure content quality, business priorities, editorial intent, or future algorithm changes. Client history is unbalanced, and early records may lack Search Console or GA4 data, so results should be interpreted with these limitations.

In [35]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.